# 05 · 송도 사건 구성과 TTC·PET 계산

[12. 결정안](../진행작업%20순서/12_사건구성과지표계산_결정안.md)과 04의 자동 등록표로 **후미추돌(M2 계열)**과 **교차(M1 계열)** 사건을 만들고, 사건마다 TTC·PET를 계산합니다. 두 유형의 지표 정의는 섞지 않습니다.

| 항목 | 후미추돌 (M2 계열) | 교차 (M1 계열) |
|---|---|---|
| 대상 | 04 등록표의 **접근로 차로**에서 사이에 다른 차 없이 앞뒤로 이어진 차량쌍(정지 차량 포함). 회전 전용차로 제외(M2)는 06에서 차로 역할로 적용 | 경로가 만나는 서로 다른 흐름(진입·진출 구역쌍)의 차량 중, 교차점을 **바로 이어 지나간** 쌍 |
| TTC | M2 식(1): 차간 간격(앞차 뒤끝−뒤차 앞끝) ÷ 닫히는 속도, 뒤차가 더 빠를 때만. 사건 최솟값 | 현재 속도·방향 유지 시 두 **차체 직사각형**이 처음 겹치는 시각(S06·S04 방식, 정확한 식). 사건 최솟값 |
| PET | 앞차 뒤끝이 지나간 위치에 뒤차 앞끝이 도착한 시간차를 매 프레임 계산한 **최솟값**(M2 식(3), Gettman) | 두 차량 **중심 경로의 교점**을 각 중심이 지난 시각의 차(점 기준, Ismail 2010) |
| 위치 | TTC는 평활 위치·속도, PET는 **원래 위치** | TTC는 평활 위치·속도, 교점·통과 시각(PET)은 **원래 위치** |

**자동 규칙 (사용자 결정 2026-09-23, 12번 기록)**
- **연속 관측**: 한 차량의 다음 관측이 **다음 프레임**(1.5프레임 이내)일 때만 이어진 것으로 봅니다. 한 프레임이라도 빠지면 끊긴 것이며 보간하지 않습니다(11 §6).
- **PET는 원래 위치**로 계산합니다. M2 p.2: PET는 속도나 미래 위치 외삽이 필요 없는 지표이고, 제작팀은 위치를 평활하지 않고 제공했습니다(Fonod p.18). 평활은 회전 경로를 곡선 안쪽으로 당겨 교점을 옮기므로(합성 검증: 반경 17m 좌회전에서 약 0.4m, PET 약 0.09초) PET에는 쓰지 않습니다.
- **속도·방향(TTC용)**: 끊기지 않은 구간 안에서 Local 위치를 가우시안 σ=14프레임으로 평활한 뒤 미분합니다(Fonod 부록 B). 1 km/h 미만에서는 방향을 새로 정하지 않고 직전·직후 방향을 씁니다. **1 km/h는 우리의 선택**입니다(이보다 느리면 위치 잡음 때문에 방향을 믿기 어렵습니다). Fonod p.20은 같은 값을 그림을 보기 좋게 하려고 속력을 거를 때 썼을 뿐, 검증된 정지 기준이 아닙니다. 제공 `Vehicle_Speed`(÷3.6)는 우리 속력과 대조만 합니다.
- **차량쌍은 같은 드론 안에서만** 만듭니다(드론 간 시계 미확인). 두 드론이 같은 순간을 찍은 경우 행이 가장 많은 드론만 남겨 같은 차량이 두 번 세어지지 않게 합니다.
- **후미 앞차는 차체가 좌우로 겹치는 가장 가까운 앞차**입니다(옆 차로 차량의 차로 오배정 건너뜀). 차체가 겹친 자료 오류(`body_overlap`)와 위치 떨림은 표시하고, 06에서는 해당 지표를 뺍니다. `position_jitter`는 보고용 종합 표시입니다.
- **사전선별 없음**: 초 기준으로 사건을 거르지 않습니다. 교차는 "바로 이어 지나간 쌍"이라는 PET 정의로만 좁힙니다.
- **계산 불가를 0이나 큰 값으로 채우지 않습니다.** 차체 겹침·치수 결측·통과 미관측은 상태로 기록합니다.

사건·지표 유효 사건·EVT 꼬리 표본은 서로 다른 수이며 모두 '상충 건수'가 아닙니다. 06에서 EVT를 적합합니다.

**2026-09-24 교차 검토 반영:** 교차 차량쌍과 교차 TTC는 보조 드론을 뺀 뒤 **같은 드론·같은 녹화 토막** 안에서만 계산합니다. 그 드론의 관측 간격이 **2.5프레임을 넘으면** 새 토막이며, 공백을 가로지른 후보 수를 기록합니다. 한 프레임 누락(관측 간격 2프레임)은 토막을 나누지 않습니다. 04 전수 800파일의 드론 관측 간격은 1프레임 14,898,334개, 2프레임 1,589개, 3–5프레임 1개, 6–30프레임 4개, 31–299프레임 52개, 300프레임 이상 1,892개였습니다. 이 분포를 근거로 정한 자동 잠정 규칙입니다. **차량 단위 연속성·평활·미분·후미 사건은 기존 1.5프레임 규칙을 유지**합니다. 같은 차량쌍이 여러 곳에서 교차하면 각 교점을 남기고, 원래 경로의 선분 번호쌍이 같은 정밀 교점만 중복 제거합니다. 떨림은 TTC·PET·후미 2D TTC별로 따로 표시합니다.

**남은 한계:** 차량이 한 대도 관측되지 않은 시간은 실제 녹화 중단과 구별할 수 없어 토막이 나뉠 수 있습니다. 떨림 시간은 관측시간과 같은 간격 상한(1.5프레임)으로 합산하므로, 떨어진 떨림 구간 사이마다 단순 프레임 수 방식보다 최대 반 프레임 크게 셀 수 있습니다.

| 코드 셀 | 하는 일 |
|---|---|
| 1–3 | 패키지·경로·입력(01·04) 확인. **2번 셀에 04 실행 ID 입력** |
| 4 | 궤적 평활·속도·치수 함수 |
| 5 | 후미추돌 사건·지표 함수 |
| 6 | 교차 사건·지표 함수 |
| 7 | 이번 실행 폴더 |
| **8** | **전체 800개 파일 처리** (가장 오래 걸림) |
| 9 | 사건 목록 합치기와 상태 요약 |
| 10 | 06으로 넘길 요약 |

## 1. 준비
scipy(가우시안 평활)가 필요합니다. desk 커널에는 설치되어 있습니다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import importlib.util
import io
import json
import math
import time
import uuid

packages = {"pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "IPython": "IPython"}
missing = [p for m, p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("별도 셀에서 설치하세요: %pip install " + " ".join(missing))
import pandas as pd
import numpy as np
from scipy.ndimage import gaussian_filter1d
from IPython.display import display
print("준비 완료 | pandas", pd.__version__, "| numpy", np.__version__)

준비 완료 | pandas 3.0.5 | numpy 2.4.6


## 2. 입력 실행·상수
`SOURCE04_ID`에는 사용자가 실행·저장한 04의 실행 ID(`20260922T182837Z_4eaba3c7`)를 넣어 두었습니다. 04를 다시 실행했다면 그 ID로 바꾸세요. 최신 폴더를 자동으로 고르지 않습니다.

- `NEXT_FRAME_MAX_S`: 다음 프레임의 정의(1.5프레임). 초 단위 상충 기준이 아닙니다.
- `RECORDING_BREAK_S`: 드론 녹화 토막을 나누는 관측 간격(2.5프레임 초과). 차량 단위 연속 규칙과 구분합니다.
- `SIGMA_FRAMES = 14`: Fonod 부록 B에서 시험 차량 실측으로 정한 평활 폭.
- `STATIONARY_MPS`: 1 km/h. 이보다 느리면 방향이 잡음에 좌우되므로 직전·직후 방향을 씁니다.
- `LEADER_SEARCH = 4`: 후미 앞차를 찾을 때, 같은 차로 라벨 안에서 앞쪽으로 최대 4대까지 보며 **좁은 차 폭의 절반 이상이 좌우로 겹치는 첫 차**를 앞차로 삼습니다(5번 셀 설명). 그 사이의 옆 차로 차량(라벨 오배정)은 건너뜁니다. 4대 안에서 찾지 못한 경우는 개수로 기록합니다.
- `MOTORCYCLE_CLASS = 3`: README의 `Vehicle_Class` 정의(0 승용·밴, 1 버스, 2 트럭, 3 오토바이).
- `JITTER_*`: **위치 떨림 구간** 판정값입니다(4번 셀 설명). 드론별 10초 구간마다, 3m/s 이상으로 움직이는 한 프레임 이동의 "원래 위치 이동거리 ÷ 평활 위치 이동거리" 중앙값을 구합니다(이동 30개 이상인 구간만). 이 값이 1.10을 넘으면 떨림 구간입니다. 2026-09-24 점검에서 정상 파일 구간은 중앙값 1.011, 상위 10%가 1.021이었고, 제공 속력과 크게 어긋나는 파일(예: 2022-10-07_H_PM1)의 문제 구간은 1.10을 넘었습니다.
- `RDP_TOLERANCE_M`, `FLOW_RESAMPLE_POINTS`: 교차점 후보를 빠르게 찾는 계산 장치입니다(평활 경로를 단순화해 후보 구간을 찾음). 최종 교점·통과 시각은 대략적인 교점 주변(`REFINE_RADIUS_M` = 5m, 평활로 생기는 어긋남 0.5m 이하보다 충분히 넓음)의 **원래 위치**로 다시 계산하며, 그 안에서 찾지 못하면 더 넓은 구간으로 다시 찾습니다. 지표 값을 정하는 기준이 아닙니다.

In [2]:
SOURCE04_ID = "20260922T182837Z_4eaba3c7"  # 04 실행(2026-09-23, 800/800 완료). 04를 다시 돌렸다면 새 ID로 바꾸세요.
if not SOURCE04_ID:
    raise ValueError("04 노트북 마지막 셀의 실행 ID를 SOURCE04_ID에 입력하세요.")
PROJECT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                if (p / "AGENTS.md").is_file() and (p / "data/raw").is_dir()), None)
if PROJECT is None:
    raise FileNotFoundError("프로젝트 또는 analysis 폴더에서 실행하세요.")
RAW = (PROJECT / "data/raw").resolve()
SOURCE01_ID = "20260916T063821Z_28f9e8ee"
SOURCE01 = (PROJECT / "data/processed/songdo_movement" / SOURCE01_ID).resolve()
SOURCE04 = (PROJECT / "data/processed/songdo_structure" / SOURCE04_ID).resolve()
OUTPUT_ROOT = (PROJECT / "data/processed/songdo_events").resolve()
if (not OUTPUT_ROOT.is_relative_to(PROJECT)
        or any(OUTPUT_ROOT.is_relative_to(p) for p in [RAW, SOURCE01, SOURCE04])):
    raise ValueError("출력 경로가 입력과 겹칩니다.")

SITES = list("ABCEFGHIJKLMNOPQRSTU")
DATES = [f"2022-10-{d:02d}" for d in range(4, 8)]
SESSIONS = [f"{p}{i}" for p in ["AM", "PM"] for i in range(1, 6)]
EXPECTED = {f"{d}_{s}_{t}": (d, s, t) for d in DATES for s in SITES for t in SESSIONS}
PROVENANCE_KEYS = ["source_file", "file_stem", "date", "site", "session"]
READ_COLUMNS = ["Vehicle_ID", "Local_Time", "Drone_ID", "Local_X", "Local_Y",
                "Vehicle_Length", "Vehicle_Width", "Vehicle_Class", "Vehicle_Speed", "Road_Section", "Lane_Number"]
STRING_COLUMNS = {c: "string" for c in
                  ["Vehicle_ID", "Local_Time", "Drone_ID", "Road_Section", "Lane_Number"]}
CLOCK_RE = r"(?:[01]\d|2[0-3]):[0-5]\d:[0-5]\d(?:\.\d{1,9})?"
UNLABELED = "<unlabeled>"
FPS = 29.97
FRAME_S = 1.0 / FPS
NEXT_FRAME_MAX_S = 1.5 * FRAME_S
RECORDING_BREAK_S = 2.5 * FRAME_S
SIGMA_FRAMES = 14
STATIONARY_MPS = 1.0 / 3.6
RDP_TOLERANCE_M = 0.05
FLOW_RESAMPLE_POINTS = 50
REFINE_RADIUS_M = 5.0
RELATIONS = {"동방향": 0.0, "측방": 90.0, "대향": 180.0}
LEADER_SEARCH = 4
JITTER_BIN_S = 10.0
JITTER_MIN_SPEED_MPS = 3.0
JITTER_MIN_STEPS = 30
JITTER_RATIO_MAX = 1.10
MOTORCYCLE_CLASS = 3
REQUIRED_RUN_RULES = {
    "recording_break_s": RECORDING_BREAK_S,
    "crossing_pairs_within_recording_chunk": True,
    "jitter_per_indicator": True,
    "crossing_dedup_key": "vehicle_pair_and_raw_segment_pair",
    "bivariate_log_space": True,  # 06에 넘길 계산 계약; 05에서는 EVT를 적합하지 않습니다.
    "crossing_ttc_within_recording_chunk": True,
}
FORBIDDEN_RUN_IDS = {"20260923T102351Z_875f4ea7", "20260924T111836Z_ce5dd4cf"}


def validate_run_rules(metadata):
    rules = metadata.get("rules", {})
    if (metadata.get("run_id") in FORBIDDEN_RUN_IDS or not isinstance(rules, dict)
            or any(rules.get(key) != value for key, value in REQUIRED_RUN_RULES.items())):
        raise ValueError("2026-09-24 교차 검토 수정 전 실행입니다. 수정된 05로 7번부터 새로 실행하세요.")


print("입력 04:", SOURCE04_ID, "| 결과 상위 폴더:", OUTPUT_ROOT)

입력 04: 20260922T182837Z_4eaba3c7 | 결과 상위 폴더: C:\Users\123\Documents\(송도) 교통 연구 논문\data\processed\songdo_events


## 3. 입력 확인과 저장 함수
01 입력 목록으로 원자료 800개가 바뀌지 않았는지 확인하고, 04가 완료된 실행인지와 등록표·좌표계 결과를 읽습니다.

In [3]:
def now_utc():
    return datetime.now(timezone.utc).isoformat()


def signature(path):
    stat = Path(path).stat()
    return {"size_bytes": str(stat.st_size), "mtime_ns": str(stat.st_mtime_ns)}


def assert_signature(path, expected):
    if signature(path) != {k: str(expected[k]) for k in ["size_bytes", "mtime_ns"]}:
        raise RuntimeError("입력 파일 크기 또는 수정 시각 변경: " + str(path))


def stable_bytes(path):
    before = signature(path)
    content = Path(path).read_bytes()
    if signature(path) != before:
        raise RuntimeError("읽는 중 파일이 바뀌었습니다: " + str(path))
    return content, {**before, "sha256": hashlib.sha256(content).hexdigest()}


def source_path(relative, base):
    path = (PROJECT / str(relative)).resolve()
    if not path.is_relative_to(base.resolve()) or not path.is_file():
        raise ValueError("허용된 입력 파일이 아닙니다: " + str(path))
    return path


def target_path(path):
    path = Path(path).resolve()
    if (not path.is_relative_to(RUN_DIR.resolve())
            or any(path.is_relative_to(p) for p in [RAW, SOURCE01, SOURCE04])):
        raise ValueError("이번 실행 폴더 밖에는 저장하지 않습니다.")
    return path


def save_csv(table, path):
    target = target_path(path)
    temporary = target.with_name(target.name + ".tmp")
    table.to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(target)


def save_json(value, path):
    target = target_path(path)
    temporary = target.with_name(target.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(target)


controls = {}
def read_control(base, name, **kwargs):
    path = base / name
    content, fingerprint = stable_bytes(path)
    controls[path.relative_to(PROJECT).as_posix()] = fingerprint
    if name.endswith(".json"):
        return json.loads(content.decode("utf-8-sig"))
    return pd.read_csv(io.BytesIO(content), keep_default_na=False, **kwargs)


meta01 = read_control(SOURCE01, "run_metadata.json")
if not (meta01.get("run_id") == SOURCE01_ID and meta01.get("status") == "completed"
        and meta01.get("completed_files") == 800):
    raise ValueError("01이 완료된 전체 800개 실행이 아닙니다.")
manifest = read_control(SOURCE01, "input_manifest.csv", dtype={"size_bytes": "string", "mtime_ns": "string"})
if len(manifest) != 800 or set(manifest["file_stem"]) != set(EXPECTED):
    raise ValueError("01 입력 목록이 800개 조합과 다릅니다.")
for item in manifest.to_dict("records"):
    assert_signature(source_path(item["source_file"], RAW), item)
progress01 = read_control(SOURCE01, "progress.csv")
expected_rows = dict(zip(progress01["file_stem"], pd.to_numeric(progress01["n_source_rows"])))

meta04 = read_control(SOURCE04, "run_metadata.json")
if not (meta04.get("run_id") == SOURCE04_ID and meta04.get("status") == "completed"
        and meta04.get("source01_run_id") == SOURCE01_ID):
    raise ValueError("04가 완료된 실행이 아니거나 01 출처가 다릅니다.")
registry_movements = read_control(SOURCE04, "registry/registry_movements.csv",
                                  dtype={"entry_section": "string", "exit_section": "string"})
registry_lanes = read_control(SOURCE04, "registry/registry_lanes.csv",
                              dtype={"section": "string", "lane": "string"})
handed04 = read_control(SOURCE04, "coordinate_handedness_by_file.csv")
progress04 = read_control(SOURCE04, "progress.csv")
if int(progress04["status"].eq("completed").sum()) != 800:
    raise ValueError("04의 800개 파일이 모두 완료되지 않았습니다.")

approach = registry_lanes.loc[registry_lanes["auto_section_role"].eq("approach")].copy()
approach["through_share"] = pd.to_numeric(approach["직진"]) / pd.to_numeric(approach["n_vehicles"])
target_lanes = {site: {(r["section"], r["lane"]): (r["auto_lane_role"], float(r["through_share"]))
                       for r in g.to_dict("records")} for site, g in approach.groupby("site")}
flow_table = registry_movements.set_index(["site", "entry_section", "exit_section"])
print("입력 확인 완료 | 후미 사건을 만들 접근 차로 수:", sum(len(v) for v in target_lanes.values()),
      "| 흐름(구역쌍) 수:", len(registry_movements))

입력 확인 완료 | 후미 사건을 만들 접근 차로 수: 730 | 흐름(구역쌍) 수: 907


## 4. 궤적 평활·속도·치수 함수
- 차량(같은 드론)의 관측을 시간순으로 놓고 **다음 프레임으로 이어지는 구간**마다 따로 평활합니다. 구간 사이를 잇거나 채우지 않습니다.
- 구간 끝은 **점대칭 확장**으로 평활합니다. Fonod는 속력을 거울 반사로 평활했지만, 위치에 거울 반사를 쓰면 끝에서 차량이 되돌아가는 모양이 되어 속도가 0으로 끌려갑니다. 점대칭 확장은 직선 추세를 보존하므로 등속 차량의 속도가 끝에서도 유지됩니다.
- 평활한 위치를 **프레임 간격(1/29.97초)**으로 미분해 속도벡터(m/s)를 얻습니다. 원자료 시각은 밀리초로 반올림돼 간격이 33·34ms로 번갈아 나오므로(04: 연속 관측의 99.98%가 한 프레임, 중앙값 33ms), 반올림된 시각으로 나누면 속도가 ±1.5% 흔들립니다. 끊기지 않은 구간은 정의상 매 프레임이 이어진 구간이므로 실제 프레임 간격을 씁니다. 길이가 1프레임뿐인 구간은 속도를 정하지 않습니다.
- 치수는 차량별로 제공된 길이·폭의 중앙값입니다. 없으면 비워 둡니다(임의 대체 없음).
- **드론 동시 촬영 중복 제거**: 한 파일에서 두 드론 이상이 같은 순간(1.5프레임 이내)을 찍었으면, 그 순간에는 파일 전체에서 행이 가장 많은 드론의 행만 남기고 나머지 드론의 행은 계산에서 뺍니다. 2026-09-24 점검: 동시 촬영 구간에서 두 드론이 같은 구역을 찍어 같은 차량이 다른 번호로 두 번 기록되었습니다(예: 2022-10-05_N_PM1, 1.5m 안에서 겹친 차량 조합 226개). 이를 두면 같은 상호작용이 두 번 사건이 됩니다. 뺀 행 수를 기록합니다. 동시 촬영은 전체 약 0.9시간입니다.
- **위치 떨림 구간**: 위 `JITTER_*` 규칙으로 구간을 찾아, 후미의 최소 TTC·최소 PET·최소 2D TTC 시각을 각각 `jitter_ttc`·`jitter_pet`·`jitter_ttc_2d`로 표시합니다. 교차는 두 통과 시각 중 하나라도 떨림 구간이면 `jitter_pet`, 최소 TTC 시각이 그 구간이면 `jitter_ttc`입니다. `position_jitter`는 이 표시들의 OR로 보고에만 씁니다. 06은 지표별 표시로 제외하고, 관측시간에서도 떨림 시간을 뺍니다. `jitter_flag_s`도 관측시간과 같은 `covered_seconds` 규칙(간격 1.5프레임 상한)을 씁니다. README도 위치·속도 추정 오차 가능성을 적고 있습니다. 2026-09-24 점검에서 떨림 구간은 위치가 한 프레임에 최대 약 1.4m씩 흔들렸고, 경로 길이가 직선 이동거리보다 26% 길었습니다.
- **관측시간**: 파일마다 모든 드론의 관측 시각을 합쳐(합집합) 덮인 시간을 셉니다. 한 파일에 드론 2~3대가 있을 때 같은 시각을 동시에 찍은 부분을 두 번 세지 않기 위해서입니다(2026-09-24 점검: 06이 드론별 시간을 더해 동시 촬영 0.9시간을 이중으로 셈). 이어진 프레임 간격은 그대로, 1.5프레임보다 긴 공백은 1.5프레임으로만 셉니다. 06은 이 값으로 시간당 건수를 계산합니다.

In [4]:
def read_file(item):
    raw_path = source_path(item["source_file"], RAW)
    assert_signature(raw_path, item)
    frame = pd.read_csv(raw_path, usecols=READ_COLUMNS, dtype=STRING_COLUMNS,
                        keep_default_na=False, na_values=[""], low_memory=False)
    assert_signature(raw_path, item)
    return prepare_frame(frame)


def prepare_frame(frame):
    frame["_source_row"] = np.arange(1, len(frame) + 1, dtype=np.int64)
    valid_id = frame["Vehicle_ID"].str.fullmatch(r"[1-9]\d*", na=False).to_numpy(dtype=bool)
    valid_clock = frame["Local_Time"].str.fullmatch(CLOCK_RE, na=False).to_numpy(dtype=bool)
    frame["_t"] = pd.to_timedelta(frame["Local_Time"].where(valid_clock),
                                  errors="coerce").dt.total_seconds().to_numpy(dtype=float)
    for name in ["Local_X", "Local_Y", "Vehicle_Length", "Vehicle_Width", "Vehicle_Class", "Vehicle_Speed"]:
        frame["_" + name] = pd.to_numeric(frame[name], errors="coerce").astype("float64")
    frame["_valid"] = (valid_id & valid_clock & np.isfinite(frame["_t"].to_numpy())
                       & np.isfinite(frame["_Local_X"].to_numpy()) & np.isfinite(frame["_Local_Y"].to_numpy()))
    frame["_vid"] = frame["Vehicle_ID"].fillna("").astype(str)
    frame["_drone"] = frame["Drone_ID"].fillna("").astype(str)
    section = frame["Road_Section"].fillna("").astype(str).str.strip()
    lane = frame["Lane_Number"].fillna("").astype(str).str.strip()
    frame["_section"] = np.where(section.eq(""), UNLABELED, section)
    frame["_lane"] = np.where(lane.eq(""), UNLABELED, lane)
    duplicated = frame.duplicated(["_vid", "_t"], keep=False).to_numpy(dtype=bool)
    frame["_dup"] = duplicated & frame["_valid"].to_numpy(dtype=bool)
    return frame


def smooth_positions(values):
    n = len(values)
    pad = min(n - 1, int(4 * SIGMA_FRAMES))
    if pad < 1:
        return values.copy()
    left = 2 * values[0] - values[pad:0:-1]
    right = 2 * values[-1] - values[-2:-pad - 2:-1]
    extended = np.concatenate([left, values, right])
    return gaussian_filter1d(extended, SIGMA_FRAMES, mode="nearest")[pad:pad + n]


def nearest_gap(values, reference):
    if len(reference) == 0:
        return np.full(len(values), np.inf)
    k = np.searchsorted(reference, values)
    left = reference[np.clip(k - 1, 0, len(reference) - 1)]
    right = reference[np.clip(k, 0, len(reference) - 1)]
    return np.minimum(np.abs(values - left), np.abs(values - right))


def secondary_drone_rows(frame):
    usable = (frame["_valid"] & ~frame["_dup"]).to_numpy()
    drone = frame["_drone"].to_numpy()
    t = frame["_t"].to_numpy(dtype=float)
    drop = np.zeros(len(frame), dtype=bool)
    ranked = pd.Series(drone[usable]).value_counts().index.tolist()
    times = {d: np.unique(t[usable & (drone == d)]) for d in ranked}
    for i, d in enumerate(ranked[1:], start=1):
        higher = np.unique(np.concatenate([times[e] for e in ranked[:i]]))
        rows = np.flatnonzero(usable & (drone == d))
        near = nearest_gap(t[rows], higher) <= NEXT_FRAME_MAX_S
        drop[rows[near]] = True
    simultaneous = float(np.unique(t[drop]).size * FRAME_S)
    return drop, {"n_drones": len(ranked), "drone_simultaneous_s": simultaneous,
                  "n_rows_secondary_drone_dropped": int(drop.sum()),
                  "observed_drone_sum_s": float(sum(covered_seconds(times[d]) for d in ranked))}


def build_tracks(frame):
    usable = frame["_valid"] & ~frame["_dup"] & ~frame["_secondary"]
    work = frame.loc[usable, ["_vid", "_drone", "_t", "_source_row", "_Local_X", "_Local_Y",
                              "_section", "_lane", "_Vehicle_Speed"]].sort_values(
        ["_vid", "_drone", "_t", "_source_row"], kind="stable").reset_index(drop=True)
    ids = work["_vid"].to_numpy()
    drones = work["_drone"].to_numpy()
    t = work["_t"].to_numpy(dtype=float)
    new_track = np.ones(len(work), dtype=bool)
    step = np.full(len(work), np.inf)
    if len(work) > 1:
        new_track[1:] = (ids[1:] != ids[:-1]) | (drones[1:] != drones[:-1])
        step[1:] = np.diff(t)
    new_segment = new_track | (step > NEXT_FRAME_MAX_S) | (step <= 0)
    work["_track"] = np.cumsum(new_track)
    work["_seg"] = np.cumsum(new_segment)
    x = work["_Local_X"].to_numpy(dtype=float)
    y = work["_Local_Y"].to_numpy(dtype=float)
    sx, sy = x.copy(), y.copy()
    vx = np.full(len(work), np.nan)
    vy = np.full(len(work), np.nan)
    bounds = np.r_[np.flatnonzero(new_segment), len(work)]
    for a, b in zip(bounds[:-1], bounds[1:]):
        if b - a >= 2:
            sx[a:b] = smooth_positions(x[a:b])
            sy[a:b] = smooth_positions(y[a:b])
            vx[a:b] = np.gradient(sx[a:b]) / FRAME_S
            vy[a:b] = np.gradient(sy[a:b]) / FRAME_S
    speed = np.hypot(vx, vy)
    heading = pd.Series(np.where(speed >= STATIONARY_MPS, np.arctan2(vy, vx), np.nan))
    heading = heading.groupby(work["_track"]).ffill().groupby(work["_track"]).bfill()
    work["_sx"], work["_sy"], work["_vx"], work["_vy"] = sx, sy, vx, vy
    work["_speed"] = speed
    work["_heading"] = heading.to_numpy(dtype=float)
    return work


def vehicle_dimensions(frame):
    valid = frame.loc[frame["_valid"], ["_vid", "_Vehicle_Length", "_Vehicle_Width", "_Vehicle_Class"]]
    sized = valid.loc[(valid["_Vehicle_Length"] > 0) & (valid["_Vehicle_Width"] > 0)]
    dims = sized.groupby("_vid").agg(length_m=("_Vehicle_Length", "median"), width_m=("_Vehicle_Width", "median"))
    labelled = valid.dropna(subset=["_Vehicle_Class"])
    classes = labelled.groupby("_vid")["_Vehicle_Class"].agg(lambda s: int(s.mode().iloc[-1]))
    moto_share = labelled["_Vehicle_Class"].eq(MOTORCYCLE_CLASS).groupby(labelled["_vid"]).mean()
    return dims.join(classes.rename("vehicle_class"), how="outer").join(moto_share.rename("motorcycle_share"), how="outer")


def jitter_windows(tracks):
    seg = tracks["_seg"].to_numpy()
    same = np.r_[False, seg[1:] == seg[:-1]]
    x, y = tracks["_Local_X"].to_numpy(dtype=float), tracks["_Local_Y"].to_numpy(dtype=float)
    sx, sy = tracks["_sx"].to_numpy(dtype=float), tracks["_sy"].to_numpy(dtype=float)
    raw = np.r_[np.nan, np.hypot(np.diff(x), np.diff(y))]
    smooth = np.r_[np.nan, np.hypot(np.diff(sx), np.diff(sy))]
    moving = same & (smooth / FRAME_S >= JITTER_MIN_SPEED_MPS)
    bins = np.floor(tracks["_t"].to_numpy(dtype=float) / JITTER_BIN_S).astype(np.int64)
    steps = pd.DataFrame({"drone": tracks["_drone"].to_numpy()[moving], "bin": bins[moving],
                          "ratio": raw[moving] / smooth[moving]})
    stats = steps.groupby(["drone", "bin"])["ratio"].agg(["median", "size"])
    checked = stats.loc[stats["size"] >= JITTER_MIN_STEPS]
    flagged = set(checked.index[checked["median"] > JITTER_RATIO_MAX])
    times = tracks[["_drone", "_t"]].drop_duplicates()
    time_bins = np.floor(times["_t"].to_numpy(dtype=float) / JITTER_BIN_S).astype(np.int64)
    in_flag = np.array([(d, b) in flagged for d, b in zip(times["_drone"].to_numpy(), time_bins)], dtype=bool)
    return flagged, {"jitter_bins_checked": int(len(checked)), "jitter_bins_flagged": int(len(flagged)),
                     "jitter_flag_s": covered_seconds(np.unique(times.loc[in_flag, "_t"].to_numpy(dtype=float)))}


def mark_jitter(table, flagged, metric_times):
    if table.empty:
        return table
    drones = table["drone_id"].astype(str).to_numpy()
    for metric, time_columns in metric_times.items():
        hit = np.zeros(len(table), dtype=bool)
        for column in time_columns:
            if column not in table.columns:
                continue
            values = pd.to_numeric(table[column], errors="coerce").to_numpy(dtype=float)
            ok = np.isfinite(values)
            bins = np.floor(np.where(ok, values, 0.0) / JITTER_BIN_S).astype(np.int64)
            hit |= ok & np.array([(d, b) in flagged for d, b in zip(drones, bins)], dtype=bool)
        table[metric] = hit
    table["position_jitter"] = table[list(metric_times)].any(axis=1)
    return table


def covered_seconds(times):
    if len(times) == 0:
        return 0.0
    return float(np.minimum(np.diff(times), NEXT_FRAME_MAX_S).sum() + FRAME_S)


def observed_time(tracks):
    return {"observed_union_s": covered_seconds(np.unique(tracks["_t"].to_numpy(dtype=float)))}


def speed_check(tracks):
    provided = tracks["_Vehicle_Speed"].to_numpy(dtype=float) / 3.6
    ours = tracks["_speed"].to_numpy(dtype=float)
    ok = np.isfinite(provided) & np.isfinite(ours)
    if ok.sum() < 3:
        return {"n_speed_compared": int(ok.sum())}
    diff = ours[ok] - provided[ok]
    return {"n_speed_compared": int(ok.sum()),
            "speed_diff_median_mps": float(np.median(diff)),
            "speed_absdiff_p50_mps": float(np.median(np.abs(diff))),
            "speed_absdiff_p95_mps": float(np.quantile(np.abs(diff), 0.95)),
            "speed_corr": float(np.corrcoef(ours[ok], provided[ok])[0, 1])}

print("궤적 함수 정의 완료")

궤적 함수 정의 완료


## 5. 후미추돌 사건·지표 함수
- 대상: 04 등록표에서 `auto_section_role = approach`인 **모든 접근 차로**. 사건마다 그 차로의 자동 역할(`lane_role`)과 직진 비율(`lane_through_share`)을 기록합니다. 04 결과에서 차로별 직진 비율에 뚜렷한 경계가 없어(공용차로가 회전 우세로 분류되는 경우 존재), M2 p.3의 "회전 전용차로 제외"는 06에서 적용합니다. 기본은 `lane_role = through_or_shared`, 감도분석은 공용차로 포함입니다.
- 차로의 진행 방향은 그 차로를 지난 모든 관측의 속도벡터 합의 방향입니다. 위치를 이 방향에 투영해 앞뒤를 정합니다.
- 매 시각 그 차로에 있는 차량을 앞뒤 순서로 세우고, 뒤차마다 **앞쪽에서 좁은 차 폭의 절반 이상이 좌우로 겹치는 가장 가까운 차**를 앞차로 짝짓습니다(정지 차량 포함). 조건은 `|좌우 간격| < max(앞차 폭, 뒤차 폭)/2`, 즉 한 차의 중심선이 다른 차의 차체를 지나는 경우입니다. M2 식(1)은 두 차가 같은 경로 위에 있다고 가정합니다. 처음에는 차체가 조금이라도 겹치면(`< (두 폭 합)/2`) 인정했지만, 독립 검토에서 폭 추정 오차 때문에 옆 차로 차량이 몇 cm 차이로 통과하는 것이 확인되었습니다(TTC 1초 미만 42건 중 20건이 겹침 0.5m 미만, 대기행렬 옆을 지나는 차량). 그래서 이 조건으로 바꿨습니다. 다음 프레임에도 같은 앞차이면 같은 사건이 이어집니다.
  - 이 조건을 둔 이유(2026-09-24 전수 점검): 원자료의 차로 번호만으로 짝지었더니 무작위 사건의 21%가 좌우로 1.5m 넘게 떨어져 있었습니다. TTC가 가장 작은 150건의 80%는 **옆 차로에 나란히 선 차량**이었습니다(좌우 1.5–5.7m, 앞뒤 중심거리 약 4m). README도 키 큰 차량의 차로 오배정 가능성을 적고 있습니다.
  - 폭이 없으면 겹침을 판정할 수 없으므로 짝짓지 않고 개수만 셉니다(임의 폭 대체 없음). 다만 앞쪽 차량의 폭만 없고, 그 파일에서 가장 넓은 차의 폭을 가정해도 조건을 만족할 수 없을 만큼 옆에 있으면 옆 차로 차량으로 보고 건너뜁니다. 이렇게 해야 폭 없는 옆 차량 하나 때문에 한 사건이 둘로 쪼개지지 않습니다.
  - 극값 순간의 좌우 간격(`lateral_at_min_ttc_m`, `lateral_at_min_pet_m`)과 두 차의 폭을 기록합니다.
  - 앞차가 그 위치를 지난 순간이 앞차의 관측 공백 안에 있으면 보간하지 않고 PET를 계산하지 않습니다(`leader_passage_in_observation_gap`).
- **상충 유형(SSAM 규칙, Gettman 2008 PDF p.46)**: SSAM은 두 차량이 사건의 시작과 끝에 같은 차로에 있으면 후미, 한 차량이라도 다른 차로로 옮기면 **차로변경 상충**으로 분류합니다. SSAM의 사건은 두 차량이 충돌 경로에 들어선 때부터 그 상황이 해소될 때까지입니다(PDF p.41). 여기서는 초 기준(SSAM의 1.5초) 없이 옮깁니다. 극값(최소 TTC·최소 PET·최소 2D TTC)이 속한 **접근 구간**(M2 식(1)의 TTC가 정의되는, 즉 뒤차가 앞차에 다가가는 연속 프레임. 극값 순간이 접근 중이 아니면 그 프레임)이 사건의 시작이나 끝까지 이어지고, 그 경계에서 앞차나 뒤차가 **같은 구역의 다른 차로에서 들어왔거나 다른 차로로 나갔다면** 그 극값은 `lane_change`입니다. 차로를 바꾸는 차는 라벨이 바뀌기 전에 먼저 옆으로 비켜나 짝이 풀리므로, 짝이 풀린 뒤 끊김 없이 같은 차로 라벨에 머물다가 처음 바뀐 라벨로 판정합니다(시작 쪽은 거꾸로). 이 사건은 후미 분석(06)에서 빼고 기록만 남깁니다. 오래 따라가다 한참 뒤에 차로를 바꾼 경우는 충돌 경로 구간이 끊기므로 후미로 남습니다. 구역을 벗어나 교차로로 들어가는 것은 SSAM에서 링크 변경에 해당하며, 두 차량의 진행각 차이가 작으면 후미이므로 그대로 둡니다.
- **접근 구간이 없는 PET도 경계의 차로 변경을 확인합니다**(2026-09-24 교차 검토 결정). 유한 TTC가 없어도 PET 극값이 사건 첫·끝 프레임이고 해당 경계에 차로 변경이 있으면 `lane_change`로 남깁니다. 사건 시작·끝 차로로 판단하는 SSAM 규칙을 유지한 것이며, 이 분류 코드는 바꾸지 않았습니다.
- **2D TTC(비교용)**: SSAM(Gettman 2008 PDF pp.38–41)과 S06(p.14)처럼 두 차체를 직사각형으로 보고, 현재 속도와 방향을 유지하면 처음 겹치는 시각을 교차 TTC와 같은 분리축 방법으로 계산합니다(`min_ttc_2d_s`, `ttc_2d_status`). 좌우 어긋남이 물리적으로 반영됩니다. 기본 후미 TTC는 주 논문 M2 식(1)이고, 2D TTC는 06에서 감도층으로 비교합니다.
- **품질 점검용 기록**(값으로 거르지 않음): 두 차량의 차종(`leader_class`·`follower_class`, 3 = 오토바이)과 `motorcycle_involved`(한 차량 관측의 절반 이상이 오토바이로 분류되면 오토바이), 최소 TTC·최소 PET가 나온 순간의 두 차량 속력, 두 궤적의 처음·마지막 관측 시각. Fonod 부록 D는 오토바이와 특수·대형 트럭의 궤적이 쪼개지거나 둘로 나뉠 수 있다고 경고합니다. 2026-09-24 점검에서 남은 극소 PET(0.02–0.2초)는 ① 긴 차량 하나가 두 대로 검출된 경우(25cm 간격으로 함께 움직이다 같은 순간 사라짐), ② 거의 정지한 대기행렬(시속 1km 안팎, 간격이 길이 추정 오차에 좌우), ③ 오토바이가 차량 옆에 붙은 경우였습니다. PET 값 자체로 거르면 EVT가 추정하려는 꼬리를 스스로 자르게 되므로, 05는 PET와 무관한 근거(차종·속력)만 기록하고 06에서 감도분석으로 씁니다.
- **차체 겹침(`body_overlap`)**: 좌우로 겹치는 두 차량의 앞뒤 간격이 0 이하인 프레임이 있으면, 차체가 서로 겹친 것이므로 물리적으로 불가능한 자료 오류(길이 과대 추정·중복 검출 등)입니다. 이런 사건은 TTC·PET 상태를 `body_overlap`으로 표시하고, 06에서 쓰지 않습니다. 값은 확인용으로 남깁니다.
- 사건이 끝난 이유를 기록합니다: 사이 차량 진입(새 앞차가 옛 앞차보다 가까움), 앞차 좌우 어긋남, 순서 역전, 앞차·뒤차의 차로 이탈 또는 관측 공백, 차로 전체의 관측 공백, 관측 끝. 같은 쌍이 다시 이웃하면 새 사건이며 재접근으로 표시합니다. 드론 관측의 처음·끝에 걸친 사건은 잘림으로 표시해 보존합니다.
- **TTC**: 간격 = (앞차 중심 − 앞차 길이/2) − (뒤차 중심 + 뒤차 길이/2). 간격>0이고 뒤차가 더 빠를 때만 간격÷닫히는 속도. 간격≤0인 프레임은 겹침 수로 셉니다(위 `body_overlap`).
- **PET**: 매 프레임 뒤차 앞끝 위치를 앞차 뒤끝이 처음 지나간 시각을 앞차 기록에서 찾아(선형보간) 그 시간차를 구하고 사건 최솟값을 씁니다. 앞차가 그 위치를 지난 순간이 관측 전이면 계산하지 않습니다.

In [5]:
def lane_end_reasons(events, slice_times, present, pair_set, last_time_of, leader_of, position_of):
    reasons = []
    for leader, follower, last in zip(events["leader_id"], events["follower_id"], events["end_time_s"]):
        k = int(np.searchsorted(slice_times, last, side="right"))
        if k >= len(slice_times):
            reasons.append("관측끝")
            continue
        t_next = slice_times[k]
        if t_next - last > NEXT_FRAME_MAX_S:
            reasons.append("차로_관측공백")
            continue
        leader_in = (t_next, leader) in present
        follower_in = (t_next, follower) in present
        if leader_in and follower_in:
            if (t_next, follower, leader) in pair_set:
                reasons.append("순서역전")
            elif (t_next, follower) in leader_of:
                s_new = position_of.get((t_next, leader_of[(t_next, follower)]))
                s_old = position_of.get((t_next, leader))
                between = s_new is not None and s_old is not None and s_new < s_old
                reasons.append("사이차량진입" if between else "앞차_좌우어긋남")
            else:
                reasons.append("앞차_좌우어긋남_또는_판정불가")
        elif not leader_in and not follower_in:
            reasons.append("두차량_이탈또는공백")
        else:
            missing, role = (leader, "앞차") if not leader_in else (follower, "뒤차")
            reasons.append(role + ("_차로이탈" if last_time_of[missing] < t_next else "_관측공백"))
    return reasons


def rear_end_events(tracks, dims, lanes_ok, item, drone_bounds):
    counts = Counter()
    if not lanes_ok:
        return pd.DataFrame(), counts
    keys = tracks["_section"].astype(str) + "|" + tracks["_lane"].astype(str)
    wanted = {f"{s}|{l}" for s, l in lanes_ok.keys()}
    lane_rows = tracks.loc[keys.isin(wanted) & np.isfinite(tracks["_vx"]) & np.isfinite(tracks["_vy"])]
    length_of = dims["length_m"].to_dict()
    width_of = dims["width_m"].to_dict()
    width_cap = float(np.nanmax(dims["width_m"].to_numpy(dtype=float))) if dims["width_m"].notna().any() else np.inf
    moto_of = dims["motorcycle_share"].ge(0.5).to_dict()
    class_of = dims["vehicle_class"].to_dict()
    track_t = tracks["_t"].to_numpy(dtype=float)
    track_label = (tracks["_section"].astype(str) + "|" + tracks["_lane"].astype(str)).to_numpy()
    track_no = tracks["_track"].to_numpy()
    starts = np.r_[0, np.flatnonzero(track_no[1:] != track_no[:-1]) + 1] if len(track_no) else np.array([], dtype=int)
    ends = np.r_[starts[1:], len(track_no)] if len(track_no) else np.array([], dtype=int)
    track_rows = dict(zip(zip(tracks["_drone"].to_numpy()[starts], tracks["_vid"].to_numpy()[starts]), zip(starts, ends)))

    def next_other_label(drone_id, vid, time, own, forward):
        """First label different from `own` that the vehicle reaches without an observation gap, going forward
        (after `time`) or backward (before `time`); None if the track ends or is interrupted first."""
        a, b = track_rows.get((drone_id, vid), (0, 0))
        if b <= a:
            return None
        k = a + int(np.searchsorted(track_t[a:b], time, side="left"))
        if k >= b or track_t[k] != time:
            return None
        labels, times = (track_label[k:b], track_t[k:b]) if forward else (track_label[a:k + 1][::-1], track_t[a:k + 1][::-1])
        other = np.flatnonzero(labels != own)
        if len(other) == 0:
            return None
        m = int(other[0])
        if np.any(np.abs(np.diff(times[:m + 1])) > NEXT_FRAME_MAX_S):
            return None
        return labels[m]

    spans = tracks.groupby(["_drone", "_vid"])["_t"].agg(["min", "max"])
    track_span = dict(zip(spans.index, zip(spans["min"].to_numpy(dtype=float), spans["max"].to_numpy(dtype=float))))
    tables = []
    for (drone, section, lane), group in lane_rows.groupby(["_drone", "_section", "_lane"], sort=True):
        velocity = group[["_vx", "_vy"]].to_numpy(dtype=float).sum(axis=0)
        norm = float(np.hypot(*velocity))
        if not norm > 0:
            counts["lane_without_direction"] += 1
            continue
        u = velocity / norm
        normal = np.array([-u[1], u[0]])
        t_all = group["_t"].to_numpy(dtype=float)
        s_all = group["_sx"].to_numpy() * u[0] + group["_sy"].to_numpy() * u[1]
        q_all = group["_sx"].to_numpy() * normal[0] + group["_sy"].to_numpy() * normal[1]
        r_all = group["_Local_X"].to_numpy() * u[0] + group["_Local_Y"].to_numpy() * u[1]
        v_all = group["_vx"].to_numpy() * u[0] + group["_vy"].to_numpy() * u[1]
        id_all = group["_vid"].to_numpy()
        order = np.lexsort((s_all, t_all))
        t, s, q, r, v, ids = (t_all[order], s_all[order], q_all[order], r_all[order],
                              v_all[order], id_all[order])
        kin = {c: group[c].to_numpy(dtype=float)[order] for c in ["_sx", "_sy", "_vx", "_vy", "_heading"]}
        w = pd.Series(ids).map(width_of).to_numpy(dtype=float)
        n_rows = len(t)
        position = np.arange(n_rows)
        leader_at = np.full(n_rows, -1)
        searching = np.ones(n_rows, dtype=bool)
        for step in range(1, LEADER_SEARCH + 1):
            j = np.minimum(position + step, n_rows - 1)
            searching &= (position + step < n_rows) & (t[j] == t)
            dq = np.abs(q[j] - q)
            known = np.isfinite(w) & np.isfinite(w[j])
            hit = searching & known & (dq < np.maximum(w, w[j]) / 2)
            clear_side = searching & ~known & np.isfinite(w) & (dq >= np.maximum(w, width_cap) / 2)
            unknown = searching & ~known & ~clear_side
            leader_at[hit] = j[hit]
            counts["pair_rows_width_missing"] += int(unknown.sum())
            searching &= ~(hit | unknown)
        counts["pair_rows_search_exhausted"] += int(searching.sum())
        k = np.flatnonzero(leader_at >= 0)
        if len(k) == 0:
            continue
        j = leader_at[k]
        counts["pair_rows"] += int(len(k))
        counts["pair_rows_skipped_side_vehicle"] += int((j - k > 1).sum())
        pairs = pd.DataFrame({"t": t[k], "follower_id": ids[k], "leader_id": ids[j],
                              "s_f": s[k], "s_l": s[j], "r_f": r[k], "v_f": v[k], "v_l": v[j],
                              "lat": q[j] - q[k], "skipped": j - k - 1,
                              **{c + "_f": kin[c][k] for c in kin}, **{c + "_l": kin[c][j] for c in kin}})
        pairs = pairs.sort_values(["leader_id", "follower_id", "t"], kind="stable").reset_index(drop=True)
        same_key = ((pairs["leader_id"].to_numpy()[1:] == pairs["leader_id"].to_numpy()[:-1])
                    & (pairs["follower_id"].to_numpy()[1:] == pairs["follower_id"].to_numpy()[:-1]))
        close_in_time = np.diff(pairs["t"].to_numpy()) <= NEXT_FRAME_MAX_S
        pairs["event"] = np.cumsum(np.r_[True, ~(same_key & close_in_time)])
        pairs["L_l"] = pairs["leader_id"].map(length_of).astype(float)
        pairs["L_f"] = pairs["follower_id"].map(length_of).astype(float)
        pairs["gap"] = (pairs["s_l"] - pairs["L_l"] / 2) - (pairs["s_f"] + pairs["L_f"] / 2)
        closing = pairs["v_f"] - pairs["v_l"]
        valid = (pairs["gap"] > 0) & (closing > 0)
        pairs["ttc"] = np.where(valid, pairs["gap"] / closing.where(valid, 1.0), np.inf)
        pairs["W_l"] = pairs["leader_id"].map(width_of).astype(float)
        pairs["W_f"] = pairs["follower_id"].map(width_of).astype(float)
        ttc2d, over2d = rectangle_ttc(
            pairs[["_sx_f", "_sy_f"]].to_numpy(), pairs[["_sx_l", "_sy_l"]].to_numpy(),
            pairs[["_vx_f", "_vy_f"]].to_numpy(), pairs[["_vx_l", "_vy_l"]].to_numpy(),
            pairs["_heading_f"].to_numpy(), pairs["_heading_l"].to_numpy(),
            pairs["L_f"].to_numpy(), pairs["W_f"].to_numpy(), pairs["L_l"].to_numpy(), pairs["W_l"].to_numpy())
        pairs["ttc2d"] = np.where(np.isfinite(ttc2d), ttc2d, np.inf)
        pairs["over2d"] = over2d
        pairs["heading_ok"] = np.isfinite(pairs["_heading_f"]) & np.isfinite(pairs["_heading_l"])
        pairs["pet"] = np.nan
        pairs["pet_before_obs"] = False
        pairs["pet_in_gap"] = False
        history_order = np.lexsort((t, ids))
        hid, ht, hs = ids[history_order], t[history_order], r[history_order]
        cuts = np.r_[0, np.flatnonzero(hid[1:] != hid[:-1]) + 1, len(hid)]
        last_time_of = {hid[a]: ht[b - 1] for a, b in zip(cuts[:-1], cuts[1:])}
        span_of = {hid[a]: (a, b) for a, b in zip(cuts[:-1], cuts[1:])}
        for leader, rows in pairs.groupby("leader_id", sort=False).groups.items():
            length_lead = length_of.get(leader, np.nan)
            if not np.isfinite(length_lead):
                continue
            a, b = span_of[leader]
            if b - a < 2:
                continue
            reach = np.maximum.accumulate(hs[a:b] - length_lead / 2)
            times = ht[a:b]
            front = (pairs.loc[rows, "r_f"] + pairs.loc[rows, "L_f"] / 2).to_numpy()
            index = np.searchsorted(reach, front, side="left")
            ok = (index > 0) & (index < len(reach)) & np.isfinite(front)
            safe = np.clip(index, 1, len(reach) - 1)
            in_gap = ok & ((times[safe] - times[safe - 1]) > NEXT_FRAME_MAX_S)
            ok &= ~in_gap
            pairs.loc[rows, "pet_in_gap"] = in_gap
            tau = np.full(len(front), np.nan)
            i1 = index[ok]
            i0 = i1 - 1
            r0, r1 = reach[i0], reach[i1]
            frac = np.where(r1 > r0, (front[ok] - r0) / np.where(r1 > r0, r1 - r0, 1.0), 0.0)
            tau[ok] = times[i0] + frac * (times[i1] - times[i0])
            pet = pairs.loc[rows, "t"].to_numpy() - tau
            pairs.loc[rows, "pet"] = np.where(np.isfinite(pet) & (pet >= 0), pet, np.nan)
            pairs.loc[rows, "pet_before_obs"] = (index == 0) & np.isfinite(front)
        grouped = pairs.groupby("event", sort=True)
        events = grouped.agg(leader_id=("leader_id", "first"), follower_id=("follower_id", "first"),
                             start_time_s=("t", "min"), end_time_s=("t", "max"), n_frames=("t", "size"),
                             leader_length_m=("L_l", "first"), follower_length_m=("L_f", "first"),
                             min_gap_m=("gap", "min"),
                             n_overlap_frames=("gap", lambda g: int((g <= 0).sum())),
                             max_abs_lateral_m=("lat", lambda x: float(np.abs(x).max())),
                             max_side_vehicles_skipped=("skipped", "max"),
                             n_overlap_2d_frames=("over2d", "sum"),
                             any_heading=("heading_ok", "any"),
                             n_ttc_frames=("ttc", lambda x: int(np.isfinite(x).sum())),
                             n_pet_frames=("pet", lambda x: int(np.isfinite(x).sum())),
                             any_pet_before_obs=("pet_before_obs", "any"),
                             any_pet_in_gap=("pet_in_gap", "any"))
        best_ttc = pairs.loc[pairs.groupby("event")["ttc"].idxmin(), ["event", "ttc", "t", "v_f", "v_l", "lat"]].set_index("event")
        found = np.isfinite(best_ttc["ttc"])
        events["min_ttc_s"] = best_ttc["ttc"].where(found)
        events["min_ttc_time_s"] = best_ttc["t"].where(found)
        events["follower_speed_at_min_ttc_mps"] = best_ttc["v_f"].where(found)
        events["leader_speed_at_min_ttc_mps"] = best_ttc["v_l"].where(found)
        events["lateral_at_min_ttc_m"] = best_ttc["lat"].where(found)
        pet_rows = pairs.loc[pairs["pet"].notna()]
        if len(pet_rows):
            best_pet = pet_rows.loc[pet_rows.groupby("event")["pet"].idxmin(),
                                    ["event", "pet", "t", "v_f", "v_l", "lat"]].set_index("event")
            events["min_pet_s"] = best_pet["pet"]
            events["min_pet_time_s"] = best_pet["t"]
            events["follower_speed_at_min_pet_mps"] = best_pet["v_f"]
            events["leader_speed_at_min_pet_mps"] = best_pet["v_l"]
            events["lateral_at_min_pet_m"] = best_pet["lat"]
        else:
            for column in ["min_pet_s", "min_pet_time_s", "follower_speed_at_min_pet_mps", "leader_speed_at_min_pet_mps",
                           "lateral_at_min_pet_m"]:
                events[column] = np.nan
        best2 = pairs.loc[pairs.groupby("event")["ttc2d"].idxmin(), ["event", "ttc2d", "t"]].set_index("event")
        found2 = np.isfinite(best2["ttc2d"])
        events["min_ttc_2d_s"] = best2["ttc2d"].where(found2)
        events["min_ttc_2d_time_s"] = best2["t"].where(found2)
        ev = pairs["event"].to_numpy()
        first_row = np.r_[True, ev[1:] != ev[:-1]]
        last_row = np.r_[ev[1:] != ev[:-1], True]

        def reach(finite):
            brk = np.r_[True, (ev[1:] != ev[:-1]) | (finite[1:] != finite[:-1])]
            run = np.cumsum(brk)
            has_first = pd.Series(first_row).groupby(run).transform("any").to_numpy()
            has_last = pd.Series(last_row).groupby(run).transform("any").to_numpy()
            return np.where(finite, has_first, first_row), np.where(finite, has_last, last_row)

        on_course = np.isfinite(pairs["ttc"].to_numpy())
        to_start, to_end = reach(on_course)
        own_label = f"{section}|{lane}"
        change_start, change_end = [], []
        for leader_id, follower_id, t0, t1 in zip(events["leader_id"], events["follower_id"],
                                                  events["start_time_s"], events["end_time_s"]):
            moved_in, moved_out = [], []
            for role, vid in (("앞차", leader_id), ("뒤차", follower_id)):  # noqa: B007
                before = next_other_label(drone, vid, t0, own_label, forward=False)
                after = next_other_label(drone, vid, t1, own_label, forward=True)
                if before is not None and before != own_label and before.split("|")[0] == section:
                    moved_in.append(role)
                if after is not None and after != own_label and after.split("|")[0] == section:
                    moved_out.append(role)
            change_start.append("+".join(moved_in))
            change_end.append("+".join(moved_out))
        events["lane_change_at_start"] = change_start
        events["lane_change_at_end"] = change_end
        starts_lc = events["lane_change_at_start"].ne("").to_numpy()
        ends_lc = events["lane_change_at_end"].ne("").to_numpy()
        position_of_event = {e: i for i, e in enumerate(events.index)}

        def conflict_type(best_rows, reach_start, reach_end):
            kinds = np.full(len(events), "", dtype=object)
            for e, row in best_rows.items():
                i = position_of_event[e]
                lane_change = (reach_end[row] and ends_lc[i]) or (reach_start[row] and starts_lc[i])
                kinds[i] = "lane_change" if lane_change else "rear_end"
            return kinds

        ttc_rows = pairs.loc[np.isfinite(pairs["ttc"])].groupby("event")["ttc"].idxmin()
        pet_rows_idx = pairs.loc[pairs["pet"].notna()].groupby("event")["pet"].idxmin()
        ttc2_rows = pairs.loc[np.isfinite(pairs["ttc2d"])].groupby("event")["ttc2d"].idxmin()
        events["ttc_conflict_type"] = conflict_type(ttc_rows, to_start, to_end)
        events["pet_conflict_type"] = conflict_type(pet_rows_idx, to_start, to_end)
        events["ttc_2d_conflict_type"] = conflict_type(ttc2_rows, to_start, to_end)
        events["leader_class"] = events["leader_id"].map(class_of)
        events["follower_class"] = events["follower_id"].map(class_of)
        events["leader_width_m"] = events["leader_id"].map(width_of)
        events["follower_width_m"] = events["follower_id"].map(width_of)
        events["motorcycle_involved"] = (events["leader_id"].map(moto_of).eq(True)
                                         | events["follower_id"].map(moto_of).eq(True))
        for role in ["leader", "follower"]:
            span = events[role + "_id"].map(lambda v: track_span.get((drone, v), (np.nan, np.nan)))
            events[role + "_track_first_s"] = [a for a, _ in span]
            events[role + "_track_last_s"] = [b for _, b in span]
        dims_ok = np.isfinite(events["leader_length_m"]) & np.isfinite(events["follower_length_m"])
        events["ttc_status"] = np.where(~dims_ok, "dimension_missing",
                                        np.where(events["min_ttc_s"].notna(), "computed", "no_collision_course"))
        events["pet_status"] = np.where(~dims_ok, "dimension_missing",
                                        np.where(events["min_pet_s"].notna(), "computed",
                                                 np.where(events["any_pet_before_obs"],
                                                          "leader_passage_before_observation",
                                                          np.where(events["any_pet_in_gap"],
                                                                   "leader_passage_in_observation_gap",
                                                                   "no_passage_in_event"))))
        events.loc[~dims_ok, ["min_gap_m", "min_ttc_s", "min_ttc_time_s", "min_pet_s", "min_pet_time_s"]] = np.nan
        overlap = dims_ok & (events["n_overlap_frames"] > 0)
        events.loc[overlap, "ttc_status"] = "body_overlap"
        events.loc[overlap, "pet_status"] = "body_overlap"
        widths_ok = events["leader_id"].map(width_of).notna() & events["follower_id"].map(width_of).notna()
        events["ttc_2d_status"] = np.select(
            [~(dims_ok & widths_ok), overlap | (events["n_overlap_2d_frames"] > 0), ~events["any_heading"],
             events["min_ttc_2d_s"].notna()],
            ["dimension_missing", "body_overlap", "heading_missing", "computed"], default="no_collision_course")
        events = events.drop(columns=["any_pet_before_obs", "any_pet_in_gap", "any_heading"]).reset_index(drop=True)
        events["re_approach"] = events.groupby(["leader_id", "follower_id"]).cumcount() > 0
        slice_times = np.unique(t)
        present = set(zip(t, ids))
        pair_set = set(zip(pairs["t"], pairs["leader_id"], pairs["follower_id"]))
        leader_of = dict(zip(zip(pairs["t"], pairs["follower_id"]), pairs["leader_id"]))
        position_of = dict(zip(zip(t, ids), s))
        events["end_reason"] = lane_end_reasons(events, slice_times, present, pair_set, last_time_of,
                                                leader_of, position_of)
        first_t, last_t = drone_bounds[drone]
        events["start_truncated"] = events["start_time_s"] <= first_t + NEXT_FRAME_MAX_S
        events["end_truncated"] = events["end_time_s"] >= last_t - NEXT_FRAME_MAX_S
        events["duration_s"] = events["end_time_s"] - events["start_time_s"]
        role, share = lanes_ok[(section, lane)]
        events.insert(0, "lane_through_share", share)
        events.insert(0, "lane_role", role)
        events.insert(0, "lane", lane)
        events.insert(0, "section", section)
        events.insert(0, "drone_id", drone)
        events.insert(0, "relationship_type", "rear_end")
        tables.append(events)
    table = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
    if len(table):
        for k in reversed(PROVENANCE_KEYS):
            table.insert(0, k, item[k])
    return table, counts

print("후미추돌 함수 정의 완료")

후미추돌 함수 정의 완료


## 6. 교차 사건·지표 함수
1. 04가 교차로 중앙 통과를 확인한 차량만 씁니다. 흐름 = (진입 구역, 진출 구역)이며 이동 라벨은 04 자동 라벨입니다.
2. 흐름마다 경로를 호 길이로 같은 간격 50점에 다시 찍어 **중앙값 경로**를 만듭니다. 두 흐름의 중앙값 경로가 만나는 점이 그 흐름쌍의 교차점 후보입니다. 교차점에서 두 흐름의 진행 방향 차이가 0°에 가장 가까우면(같은 방향으로 합류·분류) 교차로 보지 않습니다.
3. 두 흐름의 모든 차량이 그 교차점에 **가장 가까이 간 시각**을 구해 시간순으로 세웁니다(가장 가까운 점이 경로 끝이면 통과 미관측으로 제외). 서로 다른 흐름의 차량이 **같은 드론·같은 녹화 토막 안에서 바로 이어 지나간** 경우만 짝입니다. 보조 드론을 뺀 해당 드론의 관측 간격이 2.5프레임을 넘으면 새 토막이며, 최종 판정에는 두 정밀 통과 시각을 씁니다. PET의 정의(한 차가 떠난 뒤 다른 차가 도착)에 해당하는 쌍이며 PET 값 자체의 초 기준은 필요 없습니다.
4. 짝마다 두 차량 경로의 교점을 평활 경로로 대략 찾은 뒤, 그 주변의 **원래 위치**로 정밀하게 다시 구해 각 중심이 교점을 지난 시각의 차를 PET로 씁니다(점 기준). 교점이 없으면 사건으로 만들지 않고 수만 셉니다.
5. **TTC**: 두 통과가 속한 **같은 녹화 토막의 시작부터** 두 번째 차가 교점을 지나기 전까지, 두 차량이 같은 드론의 같은 시각에 함께 관측된 프레임만 씁니다. 차체 직사각형(길이×폭, 방향=평활 속도 방향)이 현재 속도·방향을 유지할 때 **처음 겹치는 시각**을 분리축 방법으로 정확히 계산합니다. 치수나 방향이 없으면 상태로 남깁니다.
   - 두 차체 직사각형이 이미 겹친 프레임이 하나라도 있으면 물리적으로 불가능하므로(치수·방향 추정 오차), TTC 상태를 `body_overlap`으로 표시하고 06에서 이 사건의 TTC를 쓰지 않습니다. 교차 PET는 중심 경로의 교점으로 계산해 차체 치수를 쓰지 않으므로 그대로 둡니다(후미 PET는 앞끝·뒤끝에 길이를 쓰므로 후미는 둘 다 제외).
6. 사건 분류: 두 흐름의 이동 라벨과 진입 방향 관계(대향·측방·동방향). M1의 대상은 `좌회전×대향직진`입니다.

In [6]:
def nearest_relation(angle_deg):
    unsigned = abs((float(angle_deg) + 180.0) % 360.0 - 180.0)
    names = list(RELATIONS)
    refs = np.array(list(RELATIONS.values()))
    return names[int(np.argmin(np.abs(unsigned - refs)))]


def rdp_indices(x, y, tolerance):
    n = len(x)
    keep = np.zeros(n, dtype=bool)
    if n == 0:
        return np.array([], dtype=int)
    keep[0] = keep[-1] = True
    stack = [(0, n - 1)]
    while stack:
        i, j = stack.pop()
        if j <= i + 1:
            continue
        dx, dy = x[j] - x[i], y[j] - y[i]
        norm = math.hypot(dx, dy)
        px, py = x[i + 1:j] - x[i], y[i + 1:j] - y[i]
        dist = np.abs(px * dy - py * dx) / norm if norm > 0 else np.hypot(px, py)
        k = int(np.argmax(dist))
        if dist[k] > tolerance:
            m = i + 1 + k
            keep[m] = True
            stack.extend([(i, m), (m, j)])
    return np.flatnonzero(keep)


def polyline_intersections(ax, ay, bx, by):
    if len(ax) < 2 or len(bx) < 2:
        return (np.array([], dtype=int),) * 2 + (np.array([]),) * 2
    rx = (ax[1:] - ax[:-1])[:, None]
    ry = (ay[1:] - ay[:-1])[:, None]
    sxd = (bx[1:] - bx[:-1])[None, :]
    syd = (by[1:] - by[:-1])[None, :]
    qx = bx[:-1][None, :] - ax[:-1][:, None]
    qy = by[:-1][None, :] - ay[:-1][:, None]
    den = rx * syd - ry * sxd
    with np.errstate(divide="ignore", invalid="ignore"):
        u = (qx * syd - qy * sxd) / den
        v = (qx * ry - qy * rx) / den
    hit = (den != 0) & (u >= 0) & (u <= 1) & (v >= 0) & (v <= 1)
    i, j = np.nonzero(hit)
    return i, j, u[i, j], v[i, j]


def resample_path(x, y, n_points):
    dist = np.r_[0.0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]
    if len(x) < 2 or dist[-1] <= 0:
        return None
    q = np.linspace(0.0, dist[-1], n_points)
    return np.interp(q, dist, x), np.interp(q, dist, y)


def raw_window(path, point, fallback):
    near = np.flatnonzero((path["x"] - point[0]) ** 2 + (path["y"] - point[1]) ** 2 <= REFINE_RADIUS_M ** 2)
    if len(near) >= 2:
        return int(near.min()), int(near.max())
    return fallback


def refine_crossing(pa, pb, wa, wb, point):
    (a0, a1), (b0, b1) = wa, wb
    ii, jj, uu, vv = polyline_intersections(pa["x"][a0:a1 + 1], pa["y"][a0:a1 + 1],
                                            pb["x"][b0:b1 + 1], pb["y"][b0:b1 + 1])
    best = None
    for i2, j2, u2, v2 in zip(ii, jj, uu, vv):
        ga, gb = a0 + i2, b0 + j2
        if pa["seg"][ga] != pa["seg"][ga + 1] or pb["seg"][gb] != pb["seg"][gb + 1]:
            continue
        x = pa["x"][ga] + u2 * (pa["x"][ga + 1] - pa["x"][ga])
        y = pa["y"][ga] + u2 * (pa["y"][ga + 1] - pa["y"][ga])
        dist = math.hypot(x - point[0], y - point[1])
        if best is None or dist < best["dist"]:
            # A vertex belongs to its preceding continuous raw segment. This
            # gives adjacent candidates one key without a distance tolerance.
            sa = ga - 1 if u2 == 0 and ga > 0 and pa["seg"][ga - 1] == pa["seg"][ga] else ga
            sb = gb - 1 if v2 == 0 and gb > 0 and pb["seg"][gb - 1] == pb["seg"][gb] else gb
            best = {"dist": dist, "x": float(x), "y": float(y),
                    "segment_a": int(sa), "segment_b": int(sb),
                    "t_a": float(pa["t"][ga] + u2 * (pa["t"][ga + 1] - pa["t"][ga])),
                    "t_b": float(pb["t"][gb] + v2 * (pb["t"][gb + 1] - pb["t"][gb])),
                    "dir_a": math.atan2(pa["y"][ga + 1] - pa["y"][ga], pa["x"][ga + 1] - pa["x"][ga]),
                    "dir_b": math.atan2(pb["y"][gb + 1] - pb["y"][gb], pb["x"][gb + 1] - pb["x"][gb])}
    return best


def exact_crossing(pa, pb, point):
    ka, kb = pa["keep"], pb["keep"]
    ia, jb, uu, _ = polyline_intersections(pa["sx"][ka], pa["sy"][ka], pb["sx"][kb], pb["sy"][kb])
    best = None
    for i, j, u in zip(ia, jb, uu):
        coarse = (pa["sx"][ka[i]] + u * (pa["sx"][ka[i + 1]] - pa["sx"][ka[i]]),
                  pa["sy"][ka[i]] + u * (pa["sy"][ka[i + 1]] - pa["sy"][ka[i]]))
        wide_a = (ka[max(i - 1, 0)], ka[min(i + 2, len(ka) - 1)])
        wide_b = (kb[max(j - 1, 0)], kb[min(j + 2, len(kb) - 1)])
        found = refine_crossing(pa, pb, raw_window(pa, coarse, wide_a), raw_window(pb, coarse, wide_b), point)
        if found is None:
            found = refine_crossing(pa, pb, wide_a, wide_b, point)
        if found is not None and (best is None or found["dist"] < best["dist"]):
            best = found
    return best


def rectangle_ttc(ca, cb, va, vb, ha, hb, la, wa, lb, wb):
    da = np.stack([np.cos(ha), np.sin(ha)], axis=1)
    na = np.stack([-np.sin(ha), np.cos(ha)], axis=1)
    db = np.stack([np.cos(hb), np.sin(hb)], axis=1)
    nb = np.stack([-np.sin(hb), np.cos(hb)], axis=1)
    lo = np.full(len(ca), -np.inf)
    hi = np.full(len(ca), np.inf)
    for axis in (da, na, db, nb):
        ra = la / 2 * np.abs(np.sum(da * axis, axis=1)) + wa / 2 * np.abs(np.sum(na * axis, axis=1))
        rb = lb / 2 * np.abs(np.sum(db * axis, axis=1)) + wb / 2 * np.abs(np.sum(nb * axis, axis=1))
        d = np.sum((cb - ca) * axis, axis=1)
        w = np.sum((vb - va) * axis, axis=1)
        reach = ra + rb
        with np.errstate(divide="ignore", invalid="ignore"):
            t1 = (-reach - d) / w
            t2 = (reach - d) / w
        still = w == 0
        low = np.where(still, np.where(np.abs(d) <= reach, -np.inf, np.inf), np.minimum(t1, t2))
        high = np.where(still, np.where(np.abs(d) <= reach, np.inf, -np.inf), np.maximum(t1, t2))
        lo = np.maximum(lo, low)
        hi = np.minimum(hi, high)
    overlap_now = (lo <= 0) & (hi >= 0)
    ttc = np.where((lo > 0) & (lo <= hi), lo, np.nan)
    return ttc, overlap_now


def crossing_ttc(pa, pb, t_until, dims, vid_a, vid_b, t_from):
    common, ia, ib = np.intersect1d(pa["t"], pb["t"], return_indices=True)
    keep = (common >= t_from) & (common <= t_until)
    ia, ib = ia[keep], ib[keep]
    out = {"n_copresent_frames": int(len(ia))}
    la, wa = dims["length_m"].get(vid_a, np.nan), dims["width_m"].get(vid_a, np.nan)
    lb, wb = dims["length_m"].get(vid_b, np.nan), dims["width_m"].get(vid_b, np.nan)
    if len(ia) == 0:
        return {**out, "ttc_status": "no_copresence"}
    if not all(np.isfinite([la, wa, lb, wb])):
        return {**out, "ttc_status": "dimension_missing"}
    ha, hb = pa["h"][ia], pb["h"][ib]
    ok = np.isfinite(ha) & np.isfinite(hb) & np.isfinite(pa["vx"][ia]) & np.isfinite(pb["vx"][ib])
    if not ok.any():
        return {**out, "ttc_status": "heading_or_velocity_missing"}
    ia, ib, ha, hb = ia[ok], ib[ok], ha[ok], hb[ok]
    ca = np.stack([pa["sx"][ia], pa["sy"][ia]], axis=1)
    cb = np.stack([pb["sx"][ib], pb["sy"][ib]], axis=1)
    va = np.stack([pa["vx"][ia], pa["vy"][ia]], axis=1)
    vb = np.stack([pb["vx"][ib], pb["vy"][ib]], axis=1)
    ttc, overlap = rectangle_ttc(ca, cb, va, vb, ha, hb, la, wa, lb, wb)
    out.update(n_overlap_frames=int(overlap.sum()), n_ttc_frames=int(np.isfinite(ttc).sum()))
    if np.isfinite(ttc).any():
        k = int(np.nanargmin(ttc))
        return {**out, "ttc_status": "computed", "min_ttc_s": float(ttc[k]),
                "min_ttc_time_s": float(pa["t"][ia][k])}
    return {**out, "ttc_status": "no_collision_course"}


def crossing_events(tracks, dims, vehicles04, item):
    counts = Counter()
    class_of = dims["vehicle_class"].to_dict()
    moto_of = dims["motorcycle_share"].ge(0.5).to_dict()
    site = item["site"]
    eligible = vehicles04.loc[vehicles04["movement_status"].eq("entry_exit_pair"),
                              ["Vehicle_ID", "entry_section", "exit_section"]].copy()
    eligible["flow"] = list(zip(eligible["entry_section"], eligible["exit_section"]))
    flow_of = dict(zip(eligible["Vehicle_ID"], eligible["flow"]))
    site_flows = flow_table.loc[site] if site in flow_table.index.get_level_values(0) else flow_table.iloc[0:0]
    flow_info = {key: (str(row["auto_movement"]), float(row["entry_heading_median_deg"]))
                 for key, row in site_flows.iterrows()}
    records = []
    # All retained observations of this drone define recording continuity,
    # including vehicles whose movement is unknown to the registry.
    recording_starts = {}
    for drone, group in tracks.groupby("_drone", sort=False):
        times = np.unique(group["_t"].to_numpy(dtype=float))
        recording_starts[drone] = times[np.r_[True, np.diff(times) > RECORDING_BREAK_S]]
    usable = tracks.loc[tracks["_vid"].isin(flow_of)]
    for drone, dgroup in usable.groupby("_drone", sort=True):
        starts = recording_starts[drone]
        paths = {}
        for vid, g in dgroup.groupby("_vid", sort=False):
            if len(g) < 2:
                continue
            x, y = g["_Local_X"].to_numpy(dtype=float), g["_Local_Y"].to_numpy(dtype=float)
            paths[vid] = {"t": g["_t"].to_numpy(dtype=float), "x": x, "y": y, "seg": g["_seg"].to_numpy(),
                          "sx": g["_sx"].to_numpy(dtype=float), "sy": g["_sy"].to_numpy(dtype=float),
                          "vx": g["_vx"].to_numpy(dtype=float), "vy": g["_vy"].to_numpy(dtype=float),
                          "h": g["_heading"].to_numpy(dtype=float),
                          "keep": rdp_indices(g["_sx"].to_numpy(dtype=float), g["_sy"].to_numpy(dtype=float), RDP_TOLERANCE_M)}
        flows = {}
        for vid in paths:
            flows.setdefault(flow_of[vid], []).append(vid)
        representative = {}
        for flow, members in flows.items():
            samples = [resample_path(paths[v]["x"], paths[v]["y"], FLOW_RESAMPLE_POINTS) for v in members]
            samples = [s for s in samples if s is not None]
            if samples:
                representative[flow] = (np.median([s[0] for s in samples], axis=0),
                                        np.median([s[1] for s in samples], axis=0))
        flow_list = sorted(representative)
        done_pairs = set()
        for i in range(len(flow_list)):
            for j in range(i + 1, len(flow_list)):
                fa, fb = flow_list[i], flow_list[j]
                ax, ay = representative[fa]
                bx, by = representative[fb]
                ii, jj, uu, vv = polyline_intersections(ax, ay, bx, by)
                for i2, j2, u2, v2 in zip(ii, jj, uu, vv):
                    angle_a = math.atan2(ay[i2 + 1] - ay[i2], ax[i2 + 1] - ax[i2])
                    angle_b = math.atan2(by[j2 + 1] - by[j2], bx[j2 + 1] - bx[j2])
                    if nearest_relation(math.degrees(angle_a - angle_b)) == "동방향":
                        counts["flow_crossing_same_direction_skipped"] += 1
                        continue
                    point = (ax[i2] + u2 * (ax[i2 + 1] - ax[i2]), ay[i2] + u2 * (ay[i2 + 1] - ay[i2]))
                    passages = []
                    for flow in (fa, fb):
                        for vid in flows[flow]:
                            p = paths[vid]
                            d2 = (p["x"] - point[0]) ** 2 + (p["y"] - point[1]) ** 2
                            k = int(np.argmin(d2))
                            if 0 < k < len(d2) - 1:
                                passages.append((p["t"][k], flow, vid))
                            else:
                                counts["passage_not_observed"] += 1
                    passages.sort()
                    for (t1, f1, v1), (t2, f2, v2x) in zip(passages[:-1], passages[1:]):
                        if f1 == f2:
                            continue
                        exact = exact_crossing(paths[v1], paths[v2x], point)
                        if exact is None:
                            counts["pair_without_exact_crossing"] += 1
                            continue
                        # Candidate times only order the search. Recording
                        # membership must use the refined passage times.
                        chunk_a = int(np.searchsorted(starts, exact["t_a"], side="right") - 1)
                        chunk_b = int(np.searchsorted(starts, exact["t_b"], side="right") - 1)
                        if chunk_a != chunk_b:
                            counts["pair_across_recording_gap_skipped"] += 1
                            continue
                        pair = tuple(sorted(((v1, exact["segment_a"]), (v2x, exact["segment_b"]))))
                        if pair in done_pairs:
                            continue
                        done_pairs.add(pair)
                        first, second = (v1, v2x) if exact["t_a"] <= exact["t_b"] else (v2x, v1)
                        t_first, t_second = sorted([exact["t_a"], exact["t_b"]])
                        mov1, h1 = flow_info.get(flow_of[v1], ("", np.nan))
                        mov2, h2 = flow_info.get(flow_of[v2x], ("", np.nan))
                        relation = nearest_relation(float(h1) - float(h2)) if np.isfinite([h1, h2]).all() else ""
                        movement_pair = "·".join(sorted([str(mov1), str(mov2)]))
                        record = {"relationship_type": "crossing", "drone_id": drone,
                                  "recording_chunk": chunk_a,
                                  "vehicle_a_id": v1, "vehicle_b_id": v2x,
                                  "raw_segment_a": exact["segment_a"], "raw_segment_b": exact["segment_b"],
                                  "flow_a": "→".join(flow_of[v1]), "flow_b": "→".join(flow_of[v2x]),
                                  "movement_a": mov1, "movement_b": mov2, "entry_relation": relation,
                                  "class_a": class_of.get(v1, np.nan), "class_b": class_of.get(v2x, np.nan),
                                  "motorcycle_involved": bool(moto_of.get(v1, False) or moto_of.get(v2x, False)),
                                  "crossing_class": movement_pair + "|" + relation,
                                  "m1_left_turn_opposing_through": bool({str(mov1), str(mov2)} == {"좌회전", "직진"}
                                                                        and relation == "대향"),
                                  "conflict_x": exact["x"], "conflict_y": exact["y"],
                                  "crossing_angle_deg": float(abs((math.degrees(exact["dir_a"] - exact["dir_b"]) + 180) % 360 - 180)),
                                  "first_vehicle_id": first, "t_first_pass_s": t_first, "t_second_pass_s": t_second,
                                  "pet_status": "computed", "pet_s": t_second - t_first}
                        record.update(crossing_ttc(paths[v1], paths[v2x], t_second, dims, v1, v2x,
                                                   t_from=starts[chunk_a]))
                        if record.get("n_overlap_frames", 0) > 0:
                            record["ttc_status"] = "body_overlap"
                        records.append(record)
    table = pd.DataFrame(records)
    if len(table):
        for k in reversed(PROVENANCE_KEYS):
            table.insert(0, k, item[k])
    return table, counts

print("교차 함수 정의 완료")

교차 함수 정의 완료


## 7. 이번 실행 폴더 만들기
매번 새 폴더를 만듭니다. 8번을 다시 시작하려면 이 셀부터 새 실행을 만드세요.

In [7]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = (OUTPUT_ROOT / RUN_ID).resolve()
if not RUN_DIR.is_relative_to(OUTPUT_ROOT) or any(RUN_DIR.is_relative_to(p) for p in [RAW, SOURCE01, SOURCE04]):
    raise ValueError("출력 폴더 경로 오류")
RUN_DIR.mkdir(parents=True, exist_ok=False)
for folder in ["rear_end", "crossing"]:
    (RUN_DIR / folder).mkdir()
run_metadata = {
    "run_id": RUN_ID, "source01_run_id": SOURCE01_ID, "source04_run_id": SOURCE04_ID,
    "decision_record": "진행작업 순서/12_사건구성과지표계산_결정안.md",
    "started_at_utc": now_utc(), "finished_at_utc": None, "status": "ready",
    "expected_files": 800, "attempted_files": 0, "completed_files": 0, "failed_files": 0,
    "source_control_fingerprints": controls, "original_files_are_read_only": True,
    "rules": {**REQUIRED_RUN_RULES, "next_frame_max_s": NEXT_FRAME_MAX_S, "sigma_frames": SIGMA_FRAMES,
              "stationary_mps": STATIONARY_MPS, "rdp_tolerance_m": RDP_TOLERANCE_M,
              "flow_resample_points": FLOW_RESAMPLE_POINTS,
              "pairs_within_same_drone_only": True, "prescreen_seconds": None,
              "rear_ttc": "M2 eq1 bumper gap / closing speed", "rear_pet": "min over frames, Gettman positional",
              "crossing_pet": "point: centre-path intersection", "crossing_ttc": "rectangles, constant velocity, exact SAT",
              "rear_leader": "nearest vehicle ahead in same lane label with lateral body overlap "
                             "|lateral| < (w_l + w_f)/2, search up to LEADER_SEARCH",
              "body_overlap": "rear: TTC and PET invalid; crossing: TTC invalid",
              "simultaneous_drones": "keep only the drone with most rows at simultaneous instants",
              "position_jitter": {"bin_s": JITTER_BIN_S, "min_speed_mps": JITTER_MIN_SPEED_MPS,
                                  "min_steps": JITTER_MIN_STEPS, "ratio_max": JITTER_RATIO_MAX},
              "observed_time": "union over drones, gaps capped at 1.5 frames",
              "zero_or_large_fill": False},
    "audit": "2026-09-24 full audit: side-by-side pairs, body overlap, duplicate drones, jitter, observation time",
}
save_json(run_metadata, RUN_DIR / "run_metadata.json")
progress_rows = []
print("이번 결과 폴더:", RUN_DIR)

이번 결과 폴더: C:\Users\123\Documents\(송도) 교통 연구 논문\data\processed\songdo_events\20260924T125033Z_ffaaff43


## 8. 전체 800개 파일 처리
**원자료를 읽는 셀입니다.** 파일마다 궤적을 평활하고 후미추돌·교차 사건과 지표를 계산해 저장합니다. 파일 오류는 기록하고 계속하며, 중단하면 완료된 파일 결과는 남습니다.

In [8]:
if run_metadata["status"] != "ready":
    raise RuntimeError("이미 시작한 실행입니다. 다시 시작하려면 7번에서 새 실행을 만드세요.")
run_metadata["status"] = "running"
save_json(run_metadata, RUN_DIR / "run_metadata.json")
vehicle_dtypes = {"Vehicle_ID": "string", "entry_section": "string", "exit_section": "string",
                  "movement_status": "string"}
started = time.perf_counter()
try:
    for index, item in enumerate(manifest.sort_values("file_stem").to_dict("records"), start=1):
        record = {key: item[key] for key in PROVENANCE_KEYS}
        file_started = time.perf_counter()
        frame = tracks = None
        try:
            frame = read_file(item)
            if len(frame) != int(expected_rows[item["file_stem"]]):
                raise ValueError("원자료 행 수가 01 기록과 다릅니다.")
            secondary, drone_info = secondary_drone_rows(frame)
            frame["_secondary"] = secondary
            tracks = build_tracks(frame)
            dims = vehicle_dimensions(frame)
            drone_bounds = tracks.groupby("_drone")["_t"].agg(["min", "max"]).apply(tuple, axis=1).to_dict()
            flagged_bins, jitter_info = jitter_windows(tracks)
            vehicles04 = pd.read_csv(SOURCE04 / "vehicles" / (item["file_stem"] + "_vehicles.csv"), dtype=vehicle_dtypes)
            rear, rear_counts = rear_end_events(tracks, dims, target_lanes.get(item["site"], {}), item, drone_bounds)
            cross, cross_counts = crossing_events(tracks, dims, vehicles04, item)
            rear = mark_jitter(rear, flagged_bins, {"jitter_ttc": ["min_ttc_time_s"],
                               "jitter_pet": ["min_pet_time_s"], "jitter_ttc_2d": ["min_ttc_2d_time_s"]})
            cross = mark_jitter(cross, flagged_bins, {"jitter_pet": ["t_first_pass_s", "t_second_pass_s"],
                                "jitter_ttc": ["min_ttc_time_s"]})
            save_csv(rear, RUN_DIR / "rear_end" / (item["file_stem"] + "_rear_end.csv"))
            save_csv(cross, RUN_DIR / "crossing" / (item["file_stem"] + "_crossing.csv"))
            record.update(status="completed", error="", n_rear_end_events=int(len(rear)),
                          n_crossing_events=int(len(cross)), n_vehicles_with_dimensions=int(dims["length_m"].notna().sum()),
                          n_segments=int(tracks["_seg"].nunique()), n_tracks=int(tracks["_track"].nunique()),
                          **speed_check(tracks), **drone_info, **jitter_info, **observed_time(tracks),
                          **{"rear_" + k: v for k, v in rear_counts.items()},
                          **{"cross_" + k: v for k, v in cross_counts.items()})
        except Exception as error:
            record.update(status="failed", error=type(error).__name__ + ": " + str(error))
            print("파일 오류:", item["file_stem"], "|", record["error"][:240])
        except BaseException as error:
            record.update(status="interrupted", error=type(error).__name__)
            raise
        finally:
            frame = tracks = None
            record["elapsed_s"] = round(time.perf_counter() - file_started, 3)
            progress_rows.append(record)
            progress = pd.DataFrame(progress_rows)
            run_metadata.update(attempted_files=len(progress),
                                completed_files=int(progress["status"].eq("completed").sum()),
                                failed_files=int(progress["status"].eq("failed").sum()),
                                elapsed_s=round(time.perf_counter() - started, 3))
            save_csv(progress, RUN_DIR / "progress.csv")
            save_json(run_metadata, RUN_DIR / "run_metadata.json")
        if index in (1, 5) or index % 20 == 0 or index == len(manifest):
            per_file = run_metadata["elapsed_s"] / index
            print(f"{index}/800개 | 완료 {run_metadata['completed_files']} | 오류 {run_metadata['failed_files']}"
                  f" | 경과 {run_metadata['elapsed_s'] / 60:.1f}분 | 파일당 {per_file:.1f}초"
                  f" | 남은 예상 {per_file * (800 - index) / 3600:.1f}시간")
    run_metadata["status"] = "files_processed" if run_metadata["completed_files"] == 800 else "incomplete_with_errors"
except BaseException as error:
    run_metadata["status"] = "interrupted"
    run_metadata["interruption"] = type(error).__name__
    raise
finally:
    run_metadata["files_finished_at_utc"] = now_utc()
    save_json(run_metadata, RUN_DIR / "run_metadata.json")
print("처리 상태:", run_metadata["status"])

1/800개 | 완료 1 | 오류 0 | 경과 0.1분 | 파일당 3.1초 | 남은 예상 0.7시간
5/800개 | 완료 5 | 오류 0 | 경과 0.4분 | 파일당 5.3초 | 남은 예상 1.2시간
20/800개 | 완료 20 | 오류 0 | 경과 2.1분 | 파일당 6.4초 | 남은 예상 1.4시간
40/800개 | 완료 40 | 오류 0 | 경과 3.1분 | 파일당 4.6초 | 남은 예상 1.0시간
60/800개 | 완료 60 | 오류 0 | 경과 3.8분 | 파일당 3.8초 | 남은 예상 0.8시간
80/800개 | 완료 80 | 오류 0 | 경과 5.3분 | 파일당 4.0초 | 남은 예상 0.8시간
100/800개 | 완료 100 | 오류 0 | 경과 9.3분 | 파일당 5.6초 | 남은 예상 1.1시간
120/800개 | 완료 120 | 오류 0 | 경과 12.5분 | 파일당 6.3초 | 남은 예상 1.2시간
140/800개 | 완료 140 | 오류 0 | 경과 14.2분 | 파일당 6.1초 | 남은 예상 1.1시간
160/800개 | 완료 160 | 오류 0 | 경과 15.3분 | 파일당 5.7초 | 남은 예상 1.0시간
180/800개 | 완료 180 | 오류 0 | 경과 18.7분 | 파일당 6.2초 | 남은 예상 1.1시간
200/800개 | 완료 200 | 오류 0 | 경과 20.5분 | 파일당 6.2초 | 남은 예상 1.0시간
220/800개 | 완료 220 | 오류 0 | 경과 22.9분 | 파일당 6.2초 | 남은 예상 1.0시간
240/800개 | 완료 240 | 오류 0 | 경과 24.1분 | 파일당 6.0초 | 남은 예상 0.9시간
260/800개 | 완료 260 | 오류 0 | 경과 25.0분 | 파일당 5.8초 | 남은 예상 0.9시간
280/800개 | 완료 280 | 오류 0 | 경과 26.5분 | 파일당 5.7초 | 남은 예상 0.8시간
300/800개 | 완료 300 | 오류 0 | 경과 30.6분 | 파일당 6.1초 

## 9. 사건 목록 합치기와 상태 요약
- 사건 수와 **지표가 계산된 사건 수**를 따로 봅니다. 계산되지 않은 이유(치수 결측·충돌 경로 없음·통과 미관측 등)가 상태로 남습니다.
- 사건이 0건인 파일(예: 교통량이 적은 지점의 교차 사건)은 이름만 출력하고 건너뜁니다.
- 커널을 다시 시작했다면 8번(약 1시간)을 다시 돌리지 말고, 1~3번 실행 후 이 셀의 `RESUME_RUN_ID`에 실행 ID를 넣습니다.
- 속도 대조: 우리가 위치로 계산한 속력과 제공 속력(÷3.6)의 차이입니다. 중앙값이 0 근처이고 상관이 높아야 정상입니다. 몇몇 파일에서 상관이 낮은 것은 위치 떨림 구간 때문입니다. 제공 속력은 떨림으로 부풀지만, 위치를 평활한 뒤 미분한 우리 속력은 몇 초 동안의 실제 이동거리와 맞습니다(2026-09-24 점검).
- 차체 겹침(`body_overlap`), 보고용 떨림 종합 표시와 지표별 떨림 표시 수, 녹화 공백 때문에 건너뛴 교차 후보 수, 드론 동시 촬영에서 뺀 행 수, 합집합 관측시간을 함께 보여줍니다. 06에서는 떨림이 표시된 지표만 제외합니다.

In [9]:
# 커널을 다시 시작했다면 1~3번 셀만 실행하고, 끝난 실행 ID를 아래에 넣은 뒤 이 셀부터 실행합니다(7·8번 생략).
RESUME_RUN_ID = ""
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
    RUN_DIR = (OUTPUT_ROOT / RUN_ID).resolve()
    run_metadata = json.loads((RUN_DIR / "run_metadata.json").read_text(encoding="utf-8"))
validate_run_rules(run_metadata)
progress = pd.read_csv(RUN_DIR / "progress.csv", keep_default_na=False)
completed = progress.loc[progress["status"].eq("completed")].copy()
if len(completed) != 800:
    print("주의: 완료 파일", len(completed), "/ 800. 완료 파일만 합칩니다.")
id_dtypes = {"leader_id": "string", "follower_id": "string", "vehicle_a_id": "string",
             "vehicle_b_id": "string", "first_vehicle_id": "string", "drone_id": "string",
             "section": "string", "lane": "string"}


def gather(kind):
    parts, no_event_files = [], []
    for stem in completed["file_stem"]:
        path = RUN_DIR / kind / f"{stem}_{kind}.csv"
        try:
            parts.append(pd.read_csv(path, dtype=id_dtypes))
        except pd.errors.EmptyDataError:
            # 사건이 0건인 파일은 열 이름도 없는 빈 CSV(BOM+줄바꿈)로 저장되어 있습니다.
            no_event_files.append(stem)
    if no_event_files:
        print(f"[{kind}] 사건 0건 파일 {len(no_event_files)}개:", ", ".join(no_event_files))
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


rear_all = gather("rear_end")
cross_all = gather("crossing")
save_csv(rear_all, RUN_DIR / "event_catalog_rear_end.csv")
save_csv(cross_all, RUN_DIR / "event_catalog_crossing.csv")
print("후미추돌 사건:", len(rear_all), "| 교차 사건:", len(cross_all))

for name, table in [("후미추돌", rear_all), ("교차", cross_all)]:
    if table.empty:
        continue
    print(f"\n[{name}] TTC 상태")
    display(table["ttc_status"].value_counts().to_frame("사건 수"))
    print(f"[{name}] PET 상태")
    display(table["pet_status"].value_counts().to_frame("사건 수"))
if not rear_all.empty:
    print("\n[후미추돌] 사건 종료 이유")
    display(rear_all["end_reason"].value_counts().to_frame("사건 수"))
    display(rear_all.groupby("site").agg(사건=("ttc_status", "size"),
                                         TTC계산=("ttc_status", lambda s: int(s.eq("computed").sum())),
                                         PET계산=("pet_status", lambda s: int(s.eq("computed").sum())),
                                         둘다=("pet_status", lambda s: int((s.eq("computed") & rear_all.loc[s.index, "ttc_status"].eq("computed")).sum()))))
if not cross_all.empty:
    print("\n[교차] 사건 분류(이동쌍|진입 관계)")
    display(cross_all["crossing_class"].value_counts().head(15).to_frame("사건 수"))
    m1 = cross_all.loc[cross_all["m1_left_turn_opposing_through"].astype(str).str.lower().eq("true")]
    print("M1 대상(좌회전×대향직진) 사건:", len(m1), "| TTC 계산:", int(m1["ttc_status"].eq("computed").sum()),
          "| PET 계산:", int(m1["pet_status"].eq("computed").sum()))
for name, table in [("후미추돌", rear_all), ("교차", cross_all)]:
    if table.empty:
        continue
    jitter = table["position_jitter"].astype(str).str.lower().eq("true")
    overlap = table["ttc_status"].eq("body_overlap") | table["pet_status"].eq("body_overlap")
    print(f"[{name}] 차체 겹침(자료 오류) {int(overlap.sum())}건 | 떨림 종합 표시(OR, 보고용) {int(jitter.sum())}건")
    for label, column in [("TTC", "jitter_ttc"), ("PET", "jitter_pet"), ("2D TTC", "jitter_ttc_2d")]:
        if column in table:
            count = table[column].astype(str).str.lower().eq("true").sum()
            print(f"  {label} 떨림 표시 {int(count)}건 → 06의 해당 지표에서만 제외")
gap_skipped = pd.to_numeric(completed.get("cross_pair_across_recording_gap_skipped",
                                        pd.Series(0, index=completed.index)), errors="coerce").fillna(0).sum()
print("[교차] 녹화 공백으로 건너뛴 후보 횟수:", int(gap_skipped), "(후보 교점별 집계)")
for c in ["n_drones", "drone_simultaneous_s", "n_rows_secondary_drone_dropped", "jitter_flag_s",
          "jitter_bins_flagged", "jitter_bins_checked", "observed_union_s", "observed_drone_sum_s"]:
    completed[c] = pd.to_numeric(completed[c], errors="coerce")
multi = completed.loc[completed["n_drones"] > 1]
print("\n[드론 동시 촬영] 드론 2대 이상 파일:", len(multi), "| 동시 촬영이 있던 파일:",
      int((multi["drone_simultaneous_s"] > 0).sum()), "| 동시 촬영 합계(시간):",
      round(float(completed["drone_simultaneous_s"].sum()) / 3600, 2), "| 뺀 보조 드론 행:",
      int(completed["n_rows_secondary_drone_dropped"].sum()))
print("[위치 떨림] 판정한 10초 구간", int(completed["jitter_bins_checked"].sum()), "| 떨림 구간",
      int(completed["jitter_bins_flagged"].sum()), "| 떨림 시간 합계(시간):",
      round(float(completed["jitter_flag_s"].sum()) / 3600, 3))
print("[관측시간] 합집합", round(float(completed["observed_union_s"].sum()) / 3600, 2), "시간 (드론별 합이었다면",
      round(float(completed["observed_drone_sum_s"].sum()) / 3600, 2), "시간)")
speed_columns = [c for c in completed.columns if c.startswith("speed_")]
for c in speed_columns:
    completed[c] = pd.to_numeric(completed[c], errors="coerce")
print("\n속도 대조(파일별 값의 중앙값)")
display(completed[speed_columns].median().round(4).to_frame("중앙값"))

[crossing] 사건 0건 파일 6개: 2022-10-04_G_AM1, 2022-10-04_G_AM4, 2022-10-05_G_AM1, 2022-10-05_G_AM3, 2022-10-06_G_AM2, 2022-10-07_G_AM4
후미추돌 사건: 481446 | 교차 사건: 82296

[후미추돌] TTC 상태


,사건 수
ttc_status,
computed,328722
no_collision_course,148111
body_overlap,4613


[후미추돌] PET 상태


,사건 수
pet_status,
computed,422025
leader_passage_before_observation,54338
body_overlap,4613
leader_passage_in_observation_gap,378
no_passage_in_event,92



[교차] TTC 상태


...

,사건,TTC계산,PET계산,둘다
site,,,,
A,18593,12036,16447,10527
B,33322,21566,28423,18385
C,10981,8554,10197,8012
E,17593,11519,15895,10240
F,17283,10097,15806,9039
G,1689,948,1495,832
H,5896,4423,5158,3950
I,16919,11268,15439,10270
J,42053,29441,35870,24631



[교차] 사건 분류(이동쌍|진입 관계)


,사건 수
crossing_class,
좌회전·직진|대향,19432
좌회전·직진|측방,17936
직진·직진|측방,17749
좌회전·좌회전|측방,16389
유턴·직진|대향,2234
유턴·좌회전|측방,1871
우회전·유턴|측방,1288
우회전·좌회전|측방,1219
직진·직진|대향,559


M1 대상(좌회전×대향직진) 사건: 19432 | TTC 계산: 6413 | PET 계산: 19432
[후미추돌] 차체 겹침(자료 오류) 4613건 | 떨림 종합 표시(OR, 보고용) 660건
  TTC 떨림 표시 428건 → 06의 해당 지표에서만 제외
  PET 떨림 표시 623건 → 06의 해당 지표에서만 제외
  2D TTC 떨림 표시 361건 → 06의 해당 지표에서만 제외
[교차] 차체 겹침(자료 오류) 36건 | 떨림 종합 표시(OR, 보고용) 59건
  TTC 떨림 표시 7건 → 06의 해당 지표에서만 제외
  PET 떨림 표시 58건 → 06의 해당 지표에서만 제외
[교차] 녹화 공백으로 건너뛴 후보 횟수: 25551 (후보 교점별 집계)

[드론 동시 촬영] 드론 2대 이상 파일: 320 | 동시 촬영이 있던 파일: 107 | 동시 촬영 합계(시간): 0.88 | 뺀 보조 드론 행: 5569137
[위치 떨림] 판정한 10초 구간 52420 | 떨림 구간 17 | 떨림 시간 합계(시간): 0.046
[관측시간] 합집합 137.26 시간 (드론별 합이었다면 138.14 시간)

속도 대조(파일별 값의 중앙값)


,중앙값
speed_diff_median_mps,-0.1326
speed_absdiff_p50_mps,0.1326
speed_absdiff_p95_mps,0.3762
speed_corr,0.9987


## 10. 06으로
06 노트북의 `SOURCE05_ID`에 아래 실행 ID를 넣고 이어서 실행합니다.

In [10]:
run_metadata.update(status="completed" if len(completed) == 800 else "incomplete",
                    finished_at_utc=now_utc(), n_rear_end_events=int(len(rear_all)),
                    n_crossing_events=int(len(cross_all)))
save_json(run_metadata, RUN_DIR / "run_metadata.json")
print(json.dumps({"실행_ID(06의 SOURCE05_ID)": RUN_ID, "상태": run_metadata["status"],
                  "완료파일": int(len(completed)), "후미추돌사건": int(len(rear_all)),
                  "교차사건": int(len(cross_all)), "결과폴더": str(RUN_DIR)},
                 ensure_ascii=False, indent=2))
print("노트북을 저장하세요.")

{
  "실행_ID(06의 SOURCE05_ID)": "20260924T125033Z_ffaaff43",
  "상태": "completed",
  "완료파일": 800,
  "후미추돌사건": 481446,
  "교차사건": 82296,
  "결과폴더": "C:\\Users\\123\\Documents\\(송도) 교통 연구 논문\\data\\processed\\songdo_events\\20260924T125033Z_ffaaff43"
}
노트북을 저장하세요.
